### 1. Prepare spaCy for lemmatization

The corpus file uses CLAWS5 tags, but spaCy uses Penn tags.
So, converting POS tags is necessary before lemmatisation with spaCy.

In [ ]:
import re
import spacy
from spacy.tokens import Doc
nlp = spacy.load('en_core_web_sm')

In [6]:
# transform POS tags
# CLAWS5 tags: https://ucrel.lancs.ac.uk/claws5tags.html
# Penn tags: https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html

# adjective
def transform_adj(claws_pos):
    adj_pos_dict = {
        'AJ0': 'JJ', 'AJC': 'JJR', 'AJS': 'JJS'
        }
    
    if claws_pos in adj_pos_dict:
        return adj_pos_dict[claws_pos]
    else:
        return None

# noun
def transform_noun(claws_pos):
    noun_pos_dict = {
        'NN0': 'NN', 'NN1': 'NN', 'NN2': 'NNS', 'NP0': 'NNP'
        }
    
    if claws_pos in noun_pos_dict:
        return noun_pos_dict[claws_pos]
    else:
        return None

# verb
def transform_verb(claws_pos):
    if re.fullmatch(r'V(B|D|H|V)B', claws_pos):
        return 'VBP'
    elif re.fullmatch(r'V(B|D|H|V)D', claws_pos):
        return 'VBD'
    elif re.fullmatch(r'V(B|D|H|V)G', claws_pos):
        return 'VBG'
    elif re.fullmatch(r'V(B|D|H|V)I', claws_pos):
        return 'VB'
    elif re.fullmatch(r'V(B|D|H|V)N', claws_pos):
        return 'VBN'
    elif re.fullmatch(r'V(B|D|H|V)Z', claws_pos):
        return 'VBZ'
    elif claws_pos == 'VM0':
        return 'MD'
    else:
        return None

In [7]:
def transform_pos(claws_pos = None):
    if claws_pos == None:
        return None
    elif claws_pos.startswith('AJ'):
        return transform_adj(claws_pos)
    elif claws_pos.startswith('N'):
        return transform_noun(claws_pos)
    elif claws_pos.startswith('V'):
        return transform_verb(claws_pos)
    else:
        return None

In [27]:
def lemmatize(word, claws_pos = None):
    # transfer POS tag
    penn_pos = transform_pos(claws_pos)

    # lemmatization
    if penn_pos == None:
            doc = Doc(nlp.vocab, words = [word], tags = None)
    else:
        doc = Doc(nlp.vocab, words = [word], tags = [penn_pos])
    
    for name, proc in nlp.pipeline:
        proc(doc)

    return doc[0].lemma_

In [ ]:
# lemmatization tests
print(lemmatize('saw', 'VVD'))
print(lemmatize('saw', 'NN1'))
print(lemmatize('saw'))

see
saw
see


### 2. Add lemmas & POS tags to the dataset

In [46]:
import os
from glob import glob

In [52]:
input_dir = 'icle_tagged'
output_dir = 'icle_lemma_pos'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [54]:
# tagged txt file looks like this:
# 0000002 010 <ICLE-JP-AI-0001.1>                         00 NULL    
# 0000002 020 I                                           00 PNP     
# 0000002 030 do                                          00 VDB     
# 0000002 040 not                                         00 XX0     
# 0000002 050 feel                                        00 VVI     
# 0000002 060 any                                         00 DT0     

def extract_word_pos(line):
    # e.g., 0000002 020 I                                           00 PNP     
    # -> {'word': 'I', 'pos': 'PNP'}
    pattern = re.compile(r'\d{7}\s\d{3}\s(\S+)\s+00\s([a-zA-Z0-9]{3,4})\s*')
    m = pattern.fullmatch(line)
    word_pos_dict = {}
    if m:
        word_pos_dict['word'] = m.group(1)
        word_pos_dict['pos'] = m.group(2)
    return word_pos_dict

def extract_doc(word_doc):
    # e.g., <ICLE-JP-AI-0001.1>
    # -> 'JPAI0001'
    doc = None
    pattern = re.compile(r'<ICLE-([a-zA-Z]{2})-([a-zA-Z]{2})[a-zA-Z]?-(\d{4}).\d>')
    m = pattern.fullmatch(word_doc)
    if m:
        doc = m.group(1) + m.group(2) + m.group(3)
    return doc

In [55]:
def setup_data(input_path):

    with open(input_path, 'r', encoding = 'utf-8') as f:
        texts = f.readlines()
    
    doc = '' # document name (e.g., JPAI0001)
    data = ''

    for line in texts[3:]: # first 3 lines are unnecessary
        if (extract_word_pos(line) != {}) and (extract_word_pos(line)['word'] != '-----'):
            word = extract_word_pos(line)['word']
            pos = extract_word_pos(line)['pos']
            lemma = lemmatize(word, pos)
            
            # metadata line
            if extract_doc(word):
                doc = extract_doc(word)
                data_line = f'{doc}\t@@{doc}\n'
            
            # text line
            else:
                data_line = f'{doc}\t{word}\t{lemma.lower()}\t{pos.lower()}\n'
            
            data += data_line
    
    return data

In [50]:
for path in glob(os.path.join(input_dir, '*_tag.txt')):
    group = os.path.basename(path)[:4].lower()
    data = setup_data(path)
    
    path = os.path.join(output_dir, ('icle_' + group + '.txt'))
    with open(path, 'w', encoding = 'utf-8') as f:
        f.write(data)